# Prompt Optimization (OPRO) for Customer-Support Routing

This notebook is designed to be completed within **60 minutes**. The lines you need to implement are marked with `#🎯TODO`, and every subsection that contains one has a 🎯 in its title.

## The situation

You run customer-support operations at **Alpenmarkt**, a (fictional) online retailer that receives thousands of customer messages every day. Today, your agents read each message and forward it to the right department by hand. Your COO wants this routing automated with a language model, and wants to know how accurate it will be before it goes live.

For example:

* A customer asking about a duplicate charge should go to the **Billing** team.
* A customer whose package has not arrived should go to the **Delivery** team.
* A customer who cannot log in should go to the **Account Support** team.
* A customer asking how a product feature works should go to the **Product Support** team.

Which requests belong to which department is already agreed. The open question is how to *instruct* the model so that it follows these business rules **as accurately as possible**. Instead of rewriting the prompt by hand until it seems to work, you will use **OPRO (Optimization by Prompting)**: a second LLM searches for a better prompt, and the data decides which one wins.

By the end of this notebook you will be able to answer:

1. How well does a simple, hand-written prompt route tickets, and where does it fail?
2. Can an LLM write a better prompt if it is given the rules and feedback on earlier attempts?
3. Does the optimized prompt still work on tickets it has never seen?

### The goal

OPRO treats prompt writing as a **search problem**: an optimizer LLM proposes candidate prompts, each candidate is scored on labelled examples, and the best prompt found so far is carried into the next round — together with a history of everything tried. The roadmap below shows how this notebook builds that loop, step by step.

The key idea is:

> **The prompt is the solution being optimized. The score is the objective. The optimizer LLM proposes candidate solutions.**

In [ ]:
#@title 🗺️ Roadmap: from a weak prompt to an optimized one (double-click to view the code) { display-mode: "form" }
import uuid
from IPython.display import HTML, display

PURPLE, PURPLE2 = '#667eea', '#764ba2'
GREEN, RED, AMBER = '#39b36a', '#e0796d', '#e0a23c'

# Each step: (icon, label, what happens, notebook part, has a 🎯 TODO?)
STAGES = [
    ("Prepare", False, [
        ("⚙️", "Setup",          "GPU + load the language model",                   "PART 1", False),
        ("📋", "The task",       "8 departments, 64 labelled messages",              "PART 2", False),
        ("✍️", "Baseline",       "Score a simple first prompt",                      "PART 3", True),
    ]),
    ("OPRO loop — one round by hand", True, [
        ("🤖", "Propose",        "The optimizer LLM reads the history and writes new candidate prompts", "PART 4", True),
        ("🧪", "Score",          "Run every candidate on the examples → accuracy, our objective",     "PART 5", False),
        ("🏆", "Keep the best",  "Carry the winner forward; add all candidates + scores to the history", "PART 5", True),
    ]),
    ("Scale up & check", False, [
        ("🔁", "Automate",       "Run several rounds, watch the score climb",        "PART 6", True),
        ("🧭", "Test & reflect", "Does it hold on unseen messages?",                 "PART 7 · SUMMARY", False),
    ]),
]


def render_roadmap(stages):
    """Builds the roadmap stepper as an HTML string.

    Args:
        stages: List of (title, is_loop, steps). Each step is a tuple
            (icon, label, blurb, part, has_todo). The loop stage is drawn
            as a dashed, repeating band.

    Returns:
        The HTML string for the roadmap widget.
    """
    uid = "rm" + uuid.uuid4().hex[:8]
    arrow = '<div class="arrow">→</div>'
    bands = []
    for title, is_loop, steps in stages:
        cards = []
        for icon, label, blurb, part, todo in steps:
            badge = '<span class="todo">🎯 TODO</span>' if todo else ''
            cards.append(f"""
          <div class="step">
            <div class="dot">{icon}</div>
            <div class="num">{part}</div>
            <div class="label">{label}</div>
            <div class="blurb">{blurb}</div>
            {badge}
          </div>""")
        row = arrow.join(cards)
        if is_loop:
            row += '<div class="back">↻<span>back to<br>Propose</span></div>'
        bands.append(f"""
      <div class="band{' loop' if is_loop else ''}">
        <div class="tag">{title}</div>
        <div class="row">{row}</div>
      </div>""")

    down = '<div class="down">↓</div>'
    return f"""
<div id="{uid}">
<style>
  #{uid} {{ font-family: system-ui, 'Segoe UI', Roboto, sans-serif; color: #2b2d42;
           border-radius: 18px; border: 1px solid #ecebff; padding: 18px 20px;
           background: linear-gradient(135deg, #f6f8ff, #fbf5ff); max-width: 820px; }}
  #{uid} h3 {{ margin: 0 0 4px 0; font-size: 18px; }}
  #{uid} .sub {{ margin: 0 0 14px 0; font-size: 13.5px; color: #5c5f77; line-height: 1.4; }}
  #{uid} .band {{ position: relative; padding: 22px 12px 12px 12px; border-radius: 16px; }}
  #{uid} .band.loop {{ border: 2px dashed {PURPLE}; background: rgba(102,126,234,.07); }}
  #{uid} .tag {{ position: absolute; top: 0; left: 12px; font-size: 11.5px; font-weight: 700;
                letter-spacing: .05em; text-transform: uppercase; color: #8a8da8; }}
  #{uid} .loop .tag {{ top: -12px; padding: 3px 12px; border-radius: 999px; color: #fff;
                      text-transform: none; letter-spacing: 0;
                      background: linear-gradient(135deg, {PURPLE}, {PURPLE2}); }}
  #{uid} .row {{ display: flex; flex-wrap: wrap; align-items: stretch; gap: 8px; }}
  #{uid} .step {{ flex: 1 1 150px; max-width: 200px; min-height: 150px; background: #fff;
                 border: 1px solid #e6e8ee; border-radius: 14px; padding: 12px 10px;
                 text-align: center; display: flex; flex-direction: column;
                 align-items: center; gap: 4px; box-sizing: border-box; }}
  #{uid} .dot {{ width: 44px; height: 44px; border-radius: 50%; font-size: 21px;
                display: flex; align-items: center; justify-content: center;
                background: linear-gradient(135deg, {PURPLE}, {PURPLE2});
                box-shadow: 0 3px 8px rgba(102,126,234,.35); }}
  #{uid} .num {{ font-size: 10.5px; letter-spacing: .06em; color: #8a8da8; font-weight: 600; }}
  #{uid} .label {{ font-weight: 700; font-size: 14.5px; }}
  #{uid} .blurb {{ font-size: 12.5px; color: #5c5f77; line-height: 1.35; }}
  #{uid} .todo {{ margin-top: auto; font-size: 10.5px; font-weight: 700; color: #8a5a00;
                 background: {AMBER}22; border: 1px solid {AMBER}99; border-radius: 999px;
                 padding: 2px 8px; }}
  #{uid} .arrow {{ align-self: center; color: {PURPLE}; font-size: 18px; font-weight: 700; }}
  #{uid} .back {{ align-self: center; display: flex; align-items: center; gap: 6px; color: {PURPLE2};
                 font-size: 26px; font-weight: 700; }}
  #{uid} .back span {{ font-size: 11px; line-height: 1.2; font-weight: 600; }}
  #{uid} .down {{ text-align: center; color: {PURPLE}; font-size: 18px; font-weight: 700;
                 margin: 6px 0; }}
  #{uid} .legend {{ margin-top: 12px; font-size: 12.5px; color: #5c5f77; }}
</style>
  <h3>🗺️ Your roadmap</h3>
  <p class="sub">Start with a prompt that sends too many tickets to the wrong team, then let an
  LLM <b>search</b> for a better one — scoring every attempt against the company's routing rules.</p>
  {down.join(bands)}
  <div class="legend"><b>🎯 TODO</b> marks the steps where you write code. The dashed box is the
  heart of OPRO: you run one round by hand in Parts 4–5, then Part 6 repeats it automatically.</div>
</div>
"""


display(HTML(render_roadmap(STAGES)))

## Part 1 — Setup: GPU and language model


A GPU is recommended because the language model will be run locally in the notebook.

### Use a T4 GPU

In Colab, choose:

**Runtime → Change runtime type**

Then set:

* **Hardware accelerator:** `T4 GPU`
* **High-RAM:** **Off**

You do **not** need a more powerful GPU for this exercise. We are using a relatively small language model, so a T4 is sufficient.

The setup cell below installs the required packages and checks that the GPU is active. Then we load the language model.

In [ ]:
# @title Setup { display-mode: "form" }
%pip install -q -U transformers accelerate
%pip uninstall -y -q torchvision

import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

print("PyTorch:", torch.__version__)
print("Device:", device)

if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected — using CPU.")

import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

# Shared colours: primary / gradient accent, and good / bad / caution
PURPLE, PURPLE2 = "#667eea", "#764ba2"
GREEN, RED, AMBER = "#39b36a", "#e0796d", "#e0a23c"

print("Setup complete. \u2713")

### Load the language model

We use `Qwen/Qwen2.5-1.5B-Instruct`, a small instruction-tuned language model.

The same model plays two conceptual roles:

- **Task model:** follows a candidate prompt and classifies a customer message.
- **Optimizer:** reads the task specification and optimization history and proposes new prompts.

They are conceptually different roles even though they use the same underlying model.

In [ ]:
# @title Loading the model
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

print("Loading:", MODEL_NAME)
print("The first run may download the model files...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype="auto"
)

model.eval()

model_device = next(model.parameters()).device

print("Model loaded.")
print("Model device:", model_device)

### A simple mental model

For the rest of the notebook, think of the model as:

```text
ask_llm(prompt) → answer
```

The cell below takes a prompt, sends it to the local language model, and returns the model’s answer as text. The cell is collapsed so the notebook stays readable — just run it. If you want to investigate how the model is actually called, double-click the cell to reveal the code.

In [ ]:
# @title Defining ask_llm function (double-click to view the code) { display-mode: "form" }
def ask_llm(prompt, max_new_tokens=80, sampling=False):

    messages = [
        {"role": "user", "content": prompt}
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_tensors="pt"
    ).to(model_device)

    if sampling:
        generation_kwargs = {
            "max_new_tokens": max_new_tokens,
            "do_sample": True,
            "temperature": 0.8,
            "top_p": 0.9
        }
    else:
        generation_kwargs = {
            "max_new_tokens": max_new_tokens,
            "do_sample": False
        }

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            **generation_kwargs
        )

    generated = outputs[0][inputs["input_ids"].shape[-1]:]

    return tokenizer.decode(
        generated,
        skip_special_tokens=True
    ).strip()

Here is how you can use the **ask_llm** function defined above to prompt the LLM for a response

In [ ]:
print(ask_llm("Say hello in one short sentence."))

## Part 2 — The routing problem: what exactly should the model do?

Your routing rules are fixed: each incoming message must be routed to one of eight departments, identified by a code.

The codes are intentionally **not descriptive**. This makes the initial prompt difficult enough that there is room for improvement.

### Task specification

| Code | Meaning |
|---|---|
| `D1` | duplicate or unexpected customer charge |
| `D2` | refund request or refund-status question |
| `D3` | package or order is late or has not arrived |
| `D4` | delivered package is damaged or contains the wrong item |
| `D5` | cannot log in or reset a password |
| `D6` | wants to change account/profile information |
| `D7` | app or website has a technical malfunction |
| `D8` | wants help understanding how a product feature works |




This gives the optimizer something useful to improve.

In [ ]:
task_specification = """
Classify each customer-support message into exactly one department code.

D1 = duplicate or unexpected customer charge
D2 = refund request or refund-status question
D3 = package/order is late or has not arrived
D4 = delivered package is damaged or contains the wrong item
D5 = cannot log in or reset a password
D6 = wants to change account/profile information
D7 = app or website has a technical malfunction
D8 = wants help understanding how a product feature works
"""

CATEGORIES = [
    "D1", "D2", "D3", "D4",
    "D5", "D6", "D7", "D8"
]

print(task_specification)

### Optimization examples

These examples are used during optimization.

Some are easy and some contain details that can make categories look similar.

In [ ]:
optimization_examples = [

    # D1 — Duplicate or unexpected charge
    ("I was charged twice for the same order.", "D1"),
    ("There are two identical payments on my statement.", "D1"),
    ("Why did I get charged twice for one purchase?", "D1"),
    ("I don't recognize this extra charge.", "D1"),
    ("The same subscription payment appears twice.", "D1"),
    ("I see a second charge that I wasn't expecting.", "D1"),
    ("My card was billed twice for the same transaction.", "D1"),
    ("I think I have an additional charge for an order I already paid for.", "D1"),

    # D2 — Refund request or refund-status question
    ("I want to get my money back for this order.", "D2"),
    ("Can I get a refund for my purchase?", "D2"),
    ("I was told my refund was processed but I haven't received it.", "D2"),
    ("When will my refund arrive?", "D2"),
    ("Please refund the payment I made.", "D2"),
    ("The refund hasn't appeared in my account yet.", "D2"),
    ("I'd like to request a reimbursement.", "D2"),
    ("My refund was approved several days ago. Can you check its status?", "D2"),

    # D3 — Package/order late or not arrived
    ("My package still hasn't arrived.", "D3"),
    ("Where is my order? It was supposed to arrive yesterday.", "D3"),
    ("The delivery is several days late.", "D3"),
    ("My order hasn't been delivered yet.", "D3"),
    ("The tracking says my package should have arrived already.", "D3"),
    ("Can you tell me why my delivery is delayed?", "D3"),
    ("I'm still waiting for the package I ordered last week.", "D3"),
    ("The courier hasn't delivered my order even though the estimated date has passed.", "D3"),

    # D4 — Damaged or wrong item delivered
    ("The item I received is broken.", "D4"),
    ("You sent me the wrong product.", "D4"),
    ("My package arrived damaged.", "D4"),
    ("The product in the box isn't the one I ordered.", "D4"),
    ("The item was damaged during delivery.", "D4"),
    ("I received a different model than the one I purchased.", "D4"),
    ("The box arrived, but the product inside is broken.", "D4"),
    ("My order arrived today but one of the items is damaged.", "D4"),

    # D5 — Login or password problem
    ("I can't log into my account.", "D5"),
    ("I forgot my password and need to reset it.", "D5"),
    ("My password isn't working anymore.", "D5"),
    ("I'm locked out of my account.", "D5"),
    ("The login page keeps rejecting my password.", "D5"),
    ("I cannot sign in even though I think my credentials are correct.", "D5"),
    ("How can I reset my account password?", "D5"),
    ("I tried several times but I still can't access my account.", "D5"),

    # D6 — Change account/profile information
    ("I want to change the name on my account.", "D6"),
    ("How do I update my phone number?", "D6"),
    ("I'd like to change the email address associated with my account.", "D6"),
    ("Can I update my profile information?", "D6"),
    ("I need to change my billing address.", "D6"),
    ("Where can I edit my account details?", "D6"),
    ("My phone number has changed and I want to update it.", "D6"),
    ("I entered the wrong name when I created my profile. How can I fix it?", "D6"),

    # D7 — Technical malfunction
    ("The website crashes whenever I try to upload a file.", "D7"),
    ("The app keeps freezing when I open it.", "D7"),
    ("I can't use the website because it keeps showing an error.", "D7"),
    ("The application crashes every time I try to check out.", "D7"),
    ("The page won't load properly.", "D7"),
    ("Something is broken on the website and I can't complete my request.", "D7"),
    ("The app gets stuck on the loading screen.", "D7"),
    ("I keep getting an error message when I try to use the service.", "D7"),

    # D8 — Product feature help
    ("How do I turn on email notifications?", "D8"),
    ("Can you explain how the reporting dashboard works?", "D8"),
    ("Where can I find the dark mode setting?", "D8"),
    ("How do I use the export feature?", "D8"),
    ("What does this feature do?", "D8"),
    ("Can you show me how to configure notifications?", "D8"),
    ("I don't understand how the reporting tool works.", "D8"),
    ("How can I use the search filters?", "D8"),
]

print("Optimization examples:", len(optimization_examples))

## Part 3 — The baseline: how well does a simple prompt route tickets?

### Start with a simple prompt

The company has already defined what each department handles. We therefore give the language model a **basic description of each department**, but keep these descriptions intentionally general.

We start with a **simple but underspecified prompt**. This gives the model some useful information about the business categories while leaving room for improvement.

The initial prompt tells the model:

* what task to perform,
* what each department generally handles,
* where the customer message appears,
* and what format to return.

However, the descriptions are **not very detailed** and do not explain how to distinguish between similar requests.

The optimizer can discover that **more precise category descriptions, clearer distinctions between departments, and better instructions** can make the prompt more effective.

In [ ]:
initial_prompt = """
Classify the customer support message into the appropriate department.

Departments:

D1: Payment-related issues
D2: Refund-related issues
D3: Delivery-related issues
D4: Issues with received products
D5: Login and access issues
D6: Account information issues
D7: Technical issues
D8: Product and feature questions

Customer message:
{message}

Return only the department code.
"""
print (initial_prompt)

### The evaluation objective

We need a way to decide whether one prompt is better than another.

Our objective is:

> **Maximize classification accuracy.**

The evaluation function runs a candidate prompt over all optimization examples.

In [ ]:
#this function checks the model’s answer and extracts one of the valid category
#codes from it
def extract_code(answer):
    answer = answer.upper().strip()

    for code in CATEGORIES:
        if code in answer:
            return code

    return None

### 🎯 What does `evaluate()` do?

The `evaluate()` function **measures how well a prompt performs**.

For each example in the dataset, it:

1. Inserts the customer message into the prompt.
2. Sends the prompt to the LLM.
3. Gets the model's prediction.
4. Compares it with the correct answer.
5. Counts the correct predictions.

It then calculates the **accuracy**:

```text
accuracy = correct predictions / total examples
````




When `return_details=True`, the function also returns the prediction and result for each example.

In OPRO, this accuracy is the **objective score** used to compare candidate prompts.

In [ ]:
def evaluate(prompt, dataset, return_details=False):

    correct = 0
    details = []

    for message, expected in dataset:
        full_prompt = prompt.replace("{message}",message)

        prediction = ask_llm(
            full_prompt,
            max_new_tokens=32,
            sampling=False
        )
        #🎯TODO: Extract the predicted code from the model's answer and compare it
        #with the ground truth (expected)
        #Solution starts here
        predicted_code = extract_code(prediction)
        is_correct = predicted_code == expected
        #🎯TODO Increment the correct variable by 1 if the prediction is correct
        if is_correct:
            correct += 1
        #Solution ends here
        details.append({
            "message": message,
            "expected": expected,
            "prediction": prediction,
            "predicted_code": predicted_code,
            "correct": is_correct
        })
    #🎯TODO: complete this line to calculate the score
    score = correct / len(dataset)

    if return_details:
        return score, details

    return score

In [ ]:
initial_score = evaluate(initial_prompt,optimization_examples)

print(f"Initial accuracy: {initial_score:.0%}")

The score for the initial prompt is not great. In the rest of this notebook we will try to increase it with a prompt optimization technique: OPRO.

### Inspect the initial predictions

A single accuracy number hides *which* tickets go wrong. Below we print a few of the misrouted messages, then plot where **every** message ended up.

In [ ]:
score, details = evaluate(initial_prompt,optimization_examples,return_details=True)

mistakes = [item for item in details if not item["correct"]]

print(f"Correct: {len(details) - len(mistakes)} of {len(details)} messages ({score:.0%})")
print(f"Misrouted: {len(mistakes)}")
print()
print("A few of the misrouted messages (at most 2 per department):")
print()

shown_per_code = {}
for item in mistakes:
    if shown_per_code.get(item["expected"], 0) >= 2:
        continue
    shown_per_code[item["expected"]] = shown_per_code.get(item["expected"], 0) + 1

    print(
        "✗ | Expected:", item["expected"],
        "| Predicted:", item["predicted_code"],
        "| Raw:", repr(item["prediction"])
    )

    print("   ",item["message"])

In [ ]:
# @title 📊 Visualization: where does the initial prompt misroute tickets? (double-click to view the code) { display-mode: "form" }
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgba

DEPARTMENT_NAMES = {
    "D1": "Duplicate charge",
    "D2": "Refund",
    "D3": "Late delivery",
    "D4": "Damaged / wrong item",
    "D5": "Login / password",
    "D6": "Account info",
    "D7": "Technical issue",
    "D8": "Feature help",
}


def plot_routing_report(details, title):
    """Plots per-department accuracy and the confusion matrix for one prompt.

    Args:
        details: Per-message results, as returned by evaluate(..., return_details=True).
        title: Figure title.
    """
    columns = CATEGORIES + (["None"] if any(d["predicted_code"] is None for d in details) else [])
    counts = np.zeros((len(CATEGORIES), len(columns)), dtype=int)
    for d in details:
        counts[CATEGORIES.index(d["expected"]), columns.index(d["predicted_code"] or "None")] += 1

    row_labels = [f"{c} · {DEPARTMENT_NAMES[c]}" for c in CATEGORIES]
    accuracy = counts.diagonal()[:len(CATEGORIES)] / counts.sum(axis=1)

    fig, (ax_bar, ax_cm) = plt.subplots(
        1, 2, figsize=(13, 4.8), gridspec_kw={"width_ratios": [1, 1.35]}
    )

    # Left: share of each department's messages routed correctly
    bar_colors = [GREEN if a >= 0.75 else AMBER if a >= 0.5 else RED for a in accuracy]
    y = np.arange(len(CATEGORIES))
    ax_bar.barh(y, 1.0, color="#f0f0f5")  # light track = 100%, so a 0% bar is still visible
    ax_bar.barh(y, accuracy, color=bar_colors)
    for yi, (a, n) in enumerate(zip(accuracy, counts.sum(axis=1))):
        ax_bar.text(a + 0.02, yi, f"{a:.0%}  ({counts[yi, yi]}/{n})", va="center", fontsize=9)
    ax_bar.set_yticks(y, row_labels)
    ax_bar.invert_yaxis()
    ax_bar.set_xlim(0, 1.25)
    ax_bar.set_xticks([0, 0.25, 0.5, 0.75, 1.0], ["0%", "25%", "50%", "75%", "100%"])
    ax_bar.set_xlabel("Messages routed to the correct department")
    ax_bar.set_title("Accuracy per department")

    # Right: confusion matrix — green diagonal = correct, red off-diagonal = misrouted
    peak = max(counts.max(), 1)
    colors = np.ones((*counts.shape, 4))
    for i in range(counts.shape[0]):
        for j in range(counts.shape[1]):
            if counts[i, j]:
                base = GREEN if columns[j] == CATEGORIES[i] else RED
                colors[i, j] = to_rgba(base, 0.2 + 0.8 * counts[i, j] / peak)
    ax_cm.imshow(colors, aspect="auto")
    for i in range(counts.shape[0]):
        for j in range(counts.shape[1]):
            if counts[i, j]:
                ax_cm.text(j, i, counts[i, j], ha="center", va="center", fontsize=10,
                           fontweight="bold")
    ax_cm.set_xticks(range(len(columns)), columns)
    ax_cm.set_yticks(range(len(CATEGORIES)), CATEGORIES)
    ax_cm.set_xticks(np.arange(-0.5, len(columns)), minor=True)
    ax_cm.set_yticks(np.arange(-0.5, len(CATEGORIES)), minor=True)
    ax_cm.grid(which="minor", color="#e6e8ee", linewidth=1)
    ax_cm.tick_params(which="minor", length=0)
    for side in ["top", "right", "bottom", "left"]:
        ax_cm.spines[side].set_visible(False)
    ax_cm.set_xlabel("Department the model chose")
    ax_cm.set_ylabel("Correct department")
    ax_cm.set_title("Where each message was routed")

    fig.suptitle(title, fontsize=13, fontweight="bold")
    plt.tight_layout(); plt.show()


plot_routing_report(details, f"Initial prompt — {score:.0%} of tickets routed correctly")

### Reading the result

The bar chart shows how reliably each department receives its own tickets; the grid shows where the rest went instead (rows = correct department, columns = the model's choice; green = correctly routed, red = misrouted).

In our run, the damage is concentrated, not spread evenly: billing, refund and delivery tickets mostly land in the right place, but **technical issues (D7) and feature questions (D8) were never routed correctly** — feature questions mostly ended up with *D4 "Issues with received products"*, and technical problems were scattered across billing and delivery. Your exact numbers may differ slightly.

For the business, this means the vague prompt is not "59% good everywhere": two teams would receive almost none of their own tickets, while others are flooded with misrouted ones. The likely cause is the department descriptions — "Technical issues" and "Product and feature questions" are too general for the model to tell apart from the others. That is precisely what a better prompt has to fix, and what OPRO will search for next.

## Part 4 — Proposing candidates: can an LLM write a better prompt?

### The optimizer proposes candidate prompts

Now we introduce the central idea. A second role, the **optimizer LLM**, writes new candidate prompts. It is not retrained: it simply reads one long instruction, the *meta-prompt*, shown below.

A candidate does **not** have to be better. We are searching through possible prompts, and the score decides.

In [ ]:
# @title 🧩 Visualization: anatomy of the optimizer's meta-prompt (double-click to view the code) { display-mode: "form" }
import html
import uuid
from IPython.display import HTML, display

# (part of the meta-prompt, excerpt as the optimizer reads it, why it is there)
META_PROMPT_PARTS = [
    ("The task specification",
     "TASK SPECIFICATION:\nD1 = duplicate or unexpected customer charge\n"
     "D2 = refund request or refund-status question\n…\n"
     "D8 = wants help understanding how a product feature works",
     "The company's precise routing rules. The optimizer sees them in full, but the task model "
     "only ever sees what the prompt passes on. That gap is what OPRO can close."),
    ("The objective",
     "OBJECTIVE:\nMaximize classification accuracy.",
     "Defines what “better” means. Every candidate is judged on this number alone."),
    ("The optimization history",
     "PREVIOUS OPTIMIZATION HISTORY:\nCandidate score: 59%\nCandidate prompt:\n"
     "Classify the customer support message…\n---\nCandidate score: 73%\n…",
     "The memory of the search: earlier prompts together with their scores (the last 8), "
     "so the optimizer can see which wording scored well and which did not."),
    ("The best prompt so far",
     "BEST PROMPT SO FAR:\nClassify the customer support message…\n\nBEST SCORE:\n59%",
     "The current champion: the prompt every new candidate has to beat."),
    ("The output rules",
     "Generate ONE new candidate prompt.\n…contain the exact placeholder {message}…\n"
     "Return ONLY the prompt between these markers:\n<PROMPT>\nyour prompt here\n</PROMPT>",
     "Makes the reply machine-readable: we can cut the candidate out and plug a customer "
     "message into it automatically."),
]


def render_meta_prompt_anatomy(parts):
    """Builds the annotated meta-prompt diagram as an HTML string.

    Args:
        parts: List of (title, excerpt, explanation) tuples, in meta-prompt order.

    Returns:
        The HTML string for the diagram.
    """
    uid = "mp" + uuid.uuid4().hex[:8]
    rows = "".join(f"""
    <div class="part">
      <div class="excerpt"><span class="badge">{n}</span><pre>{html.escape(excerpt)}</pre></div>
      <div class="why"><b>{html.escape(title)}</b><br>{html.escape(why)}</div>
    </div>""" for n, (title, excerpt, why) in enumerate(parts, start=1))

    return f"""
<div id="{uid}">
<style>
  #{uid} {{ font-family: system-ui, 'Segoe UI', Roboto, sans-serif; color: #2b2d42;
           border-radius: 18px; border: 1px solid #ecebff; padding: 18px 20px;
           background: linear-gradient(135deg, #f6f8ff, #fbf5ff); max-width: 900px; }}
  #{uid} h3 {{ margin: 0 0 4px 0; font-size: 18px; }}
  #{uid} .sub {{ margin: 0 0 14px 0; font-size: 13.5px; color: #5c5f77; line-height: 1.4; }}
  #{uid} .part {{ display: flex; flex-wrap: wrap; gap: 10px; margin-bottom: 10px; }}
  #{uid} .excerpt {{ flex: 3 1 320px; position: relative; background: #fff; min-width: 0;
                    border: 1px solid #e6e8ee; border-left: 4px solid {PURPLE};
                    border-radius: 14px; padding: 10px 12px 10px 44px; }}
  #{uid} .badge {{ position: absolute; left: 10px; top: 10px; width: 24px; height: 24px;
                  border-radius: 50%; color: #fff; font-size: 12.5px; font-weight: 700;
                  display: flex; align-items: center; justify-content: center;
                  background: linear-gradient(135deg, {PURPLE}, {PURPLE2}); }}
  #{uid} pre {{ margin: 0; white-space: pre-wrap; word-break: break-word; font-size: 12px;
               line-height: 1.45; font-family: ui-monospace, Menlo, Consolas, monospace;
               background: none; color: #2b2d42; }}
  #{uid} .why {{ flex: 2 1 220px; font-size: 13px; line-height: 1.45; color: #4a4d63;
                align-self: center; }}
  #{uid} .why b {{ color: #2b2d42; }}
  #{uid} .foot {{ margin-top: 6px; font-size: 12.5px; color: #5c5f77; line-height: 1.45; }}
</style>
  <h3>🧩 What the optimizer reads before writing a new prompt</h3>
  <p class="sub">The optimizer LLM is never retrained. Everything it knows about the problem comes
  from one long text, the <b>meta-prompt</b>, rebuilt every time it proposes a candidate:</p>
  {rows}
  <div class="foot">🎲 The optimizer samples its answer (temperature 0.8), so the <i>same</i>
  meta-prompt produces a <i>different</i> candidate each time. That randomness is what lets OPRO
  explore several prompts per round instead of repeating one.</div>
</div>
"""


display(HTML(render_meta_prompt_anatomy(META_PROMPT_PARTS)))

The following function formats the history of previous prompts and their scores
into a readable text that can be given to the optimizer LLM.

In [ ]:
# @title Format the history

def format_history(history, max_items=8):

    recent = history[-max_items:]
    parts = []

    for item in recent:
        parts.append("Candidate score:\n"+ f"{item['score']:.0%}"+ "\n\nCandidate prompt:\n"+ item["prompt"])

    return "\n---\n".join(parts)

### Generate multiple candidates

We generate several candidates per round.

Why? Because an optimization process is a **search**: we don't know in advance which wording will work best, so we try several and let the score decide. The diagram below shows one round of that search.

In [ ]:
# @title 🧩 Visualization: one round of the search (double-click to view the code) { display-mode: "form" }
import uuid
from IPython.display import HTML, display

# Illustrative scores for four candidates from one round (not computed here)
EXAMPLE_CANDIDATES = [("A", 0.61), ("B", 0.73), ("C", 0.78), ("D", 0.66)]
EXAMPLE_BEST_SO_FAR = 0.59


def render_search_round(candidates, best_so_far):
    """Builds the 'one round of the search' fan-out diagram as an HTML string.

    Args:
        candidates: List of (name, score) tuples, one per candidate prompt.
        best_so_far: Score of the best prompt before this round.

    Returns:
        The HTML string for the diagram.
    """
    uid = "fo" + uuid.uuid4().hex[:8]
    winner, top = max(candidates, key=lambda c: c[1])
    cards = "".join(f"""
      <div class="cand{' win' if name == winner else ''}">
        <div class="arrow">↓</div>
        <div class="card">
          <div class="pname">📝 Prompt {name}</div>
          <div class="eval">evaluate on all examples</div>
          <div class="bar"><span style="width:{score * 100:.0f}%"></span></div>
          <div class="score">{score:.0%}{' 🏆' if name == winner else ''}</div>
        </div>
        <div class="arrow">↓</div>
      </div>""" for name, score in candidates)

    verdict = (f"Prompt {winner} ({top:.0%}) beats the best so far ({best_so_far:.0%}) "
               f"and becomes the new best prompt."
               if top > best_so_far else
               f"No candidate beats the best so far ({best_so_far:.0%}), so it stays.")

    return f"""
<div id="{uid}">
<style>
  #{uid} {{ font-family: system-ui, 'Segoe UI', Roboto, sans-serif; color: #2b2d42;
           border-radius: 18px; border: 1px solid #ecebff; padding: 18px 20px;
           background: linear-gradient(135deg, #f6f8ff, #fbf5ff); max-width: 820px; }}
  #{uid} .box {{ margin: 0 auto; max-width: 460px; text-align: center; border-radius: 14px;
                padding: 10px 14px; color: #fff; font-size: 14px;
                background: linear-gradient(135deg, {PURPLE}, {PURPLE2}); }}
  #{uid} .box small {{ display: block; opacity: .9; font-size: 12px; margin-top: 2px; }}
  #{uid} .box.keep {{ background: #fff; color: #2b2d42; border: 2px solid {GREEN}; }}
  #{uid} .box.keep small {{ color: #4a4d63; opacity: 1; }}
  #{uid} .row {{ display: flex; flex-wrap: wrap; gap: 10px; justify-content: center; }}
  #{uid} .cand {{ flex: 1 1 140px; max-width: 180px; text-align: center; }}
  #{uid} .arrow {{ color: {PURPLE}; font-size: 18px; font-weight: 700; margin: 4px 0; }}
  #{uid} .card {{ background: #fff; border: 1px solid #e6e8ee; border-radius: 14px;
                 padding: 10px; }}
  #{uid} .win .card {{ border: 2px solid {GREEN}; box-shadow: 0 3px 10px rgba(57,179,106,.25); }}
  #{uid} .pname {{ font-weight: 700; font-size: 14px; }}
  #{uid} .eval {{ font-size: 11.5px; color: #8a8da8; margin: 2px 0 8px 0; }}
  #{uid} .bar {{ height: 8px; border-radius: 999px; background: #f0f0f5; overflow: hidden; }}
  #{uid} .bar span {{ display: block; height: 100%; background: #b8bccf; }}
  #{uid} .win .bar span {{ background: {GREEN}; }}
  #{uid} .score {{ margin-top: 6px; font-size: 16px; font-weight: 700; }}
  #{uid} .foot {{ margin-top: 12px; font-size: 12px; color: #8a8da8; text-align: center; }}
</style>
  <div class="box">🤖 <b>Optimizer LLM</b>
    <small>reads the same meta-prompt, but samples a different prompt each time</small></div>
  <div class="row">{cards}</div>
  <div class="box keep">🏆 <b>Keep the best</b><small>{verdict}</small></div>
  <div class="foot">Illustrative numbers. Every candidate, good or bad, is also added to the
  history, so the optimizer can learn from it in the next round.</div>
</div>
"""


display(HTML(render_search_round(EXAMPLE_CANDIDATES, EXAMPLE_BEST_SO_FAR)))

The following function cleans the optimizer's reply and extracts only the actual candidate prompt, removing extra commentary or formatting such as code fences. It is pure text clean-up, so the cell is collapsed — just run it, or double-click it to see the code.

In [ ]:
# @title Clean the optimizer's reply (double-click to view the code) { display-mode: "form" }
import re

def clean_candidate(candidate):
    candidate = candidate.strip()

    # 1. Complete <PROMPT>...</PROMPT> pairs: prefer the one that contains {message}
    #    (the first pair may just be the template echoed back: "your prompt here")
    pairs = re.findall(
        r"<PROMPT>(.*?)</PROMPT>",
        candidate,
        flags=re.DOTALL | re.IGNORECASE
    )

    if pairs:
        with_slot = [p for p in pairs if "{message}" in p]
        candidate = (with_slot or pairs)[-1]

    # 2. Remove any leftover, unpaired tags wherever they are
    candidate = re.sub(r"</?PROMPT>", "", candidate, flags=re.IGNORECASE)

    # 3. Remove code-fence lines anywhere, whatever their label (```, ```text, ```plaintext, ...)
    candidate = re.sub(r"^\s*```[\w-]*\s*$", "", candidate, flags=re.MULTILINE)

    # 4. Remove a first line that is only a label such as "prompt" or "plaintext"
    candidate = re.sub(
        r"^\s*(?:candidate |new )?(?:prompt|plaintext|text|markdown)\s*:?\s*\n",
        "",
        candidate,
        flags=re.IGNORECASE
    )

    # 5. Tidy up blank lines left behind by the removals
    candidate = re.sub(r"\n{3,}", "\n\n", candidate)

    return candidate.strip()

The following function generates the candidate prompts

In [ ]:
def generate_candidates(
    history,
    best_prompt,
    best_score,
    number_of_candidates=4
):

    history_text = format_history(history)
    candidates = []

    for candidate_number in range(1,number_of_candidates + 1):
        meta_prompt = f"""
We need to optimize a prompt for customer-support
department classification.

TASK SPECIFICATION:
{task_specification}

OBJECTIVE:
Maximize classification accuracy.

PREVIOUS OPTIMIZATION HISTORY:
--- START HISTORY ---
{history_text}
--- END HISTORY ---

BEST PROMPT SO FAR:
--- START BEST PROMPT ---
{best_prompt}
--- END BEST PROMPT ---

BEST SCORE:
{best_score:.0%}

Generate ONE new candidate prompt.

The candidate prompt must:
1. explain the meaning of D1 through D8
2. contain the exact placeholder {{message}}
3. tell the task model to return exactly one code
4. contain no commentary or explanation outside the prompt

VERY IMPORTANT:
Return ONLY the prompt between these markers:

<PROMPT>
your prompt here
</PROMPT>
"""

        raw_candidate = ask_llm(meta_prompt,max_new_tokens=350,sampling=True)
        candidate = clean_candidate(raw_candidate)

        if "{message}" in candidate:
            candidates.append(candidate)
        else:
            print(f"Candidate {candidate_number} rejected: "
            "missing {message}")

    return candidates

### 🎯 Generate the first batch

We start from the weak initial prompt.

The optimizer can now propose alternatives.

In [ ]:
history = [
    {
        "round": 0,
        "candidate_number": 0,
        "prompt": initial_prompt,
        "score": initial_score
    }
]
#🎯TODO : define the best prompt and score
#Solution starts here
best_prompt = initial_prompt
best_score = initial_score
#Solution ends here

candidates = generate_candidates(
    history,
    best_prompt,
    best_score,
    number_of_candidates=4
)

print(
    "Valid candidates generated:",
    len(candidates)
)

## Part 5 — Scoring and selecting: which candidate do we keep?

### Evaluate every candidate

Notice the separation of roles:

**Optimizer:** generates candidate prompts.

**Evaluator:** measures candidate performance.

In [ ]:
candidate_results = []

for i, candidate in enumerate(candidates,start=1):
    score = evaluate(candidate,optimization_examples)

    result = {
        "round": 1,
        "candidate_number": i,
        "prompt": candidate,
        "score": score
    }

    candidate_results.append(result)

    print(
        f"Candidate {i}: {score:.0%}"
    )

In [ ]:
# @title 📊 Visualization: round-1 candidates vs. the initial prompt (double-click to view the code) { display-mode: "form" }
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.ticker import PercentFormatter


def plot_candidates_vs_baseline(candidate_results, baseline_score):
    """Plots each candidate's accuracy as a bar against the baseline prompt.

    Args:
        candidate_results: List of dicts with "candidate_number" and "score".
        baseline_score: Accuracy of the prompt the candidates should beat.
    """
    names = [f"Candidate {r['candidate_number']}" for r in candidate_results]
    scores = [r["score"] for r in candidate_results]
    colors = [GREEN if s > baseline_score else RED if s < baseline_score else AMBER
              for s in scores]

    fig, ax = plt.subplots(figsize=(7.5, 4.2))
    bars = ax.bar(names, scores, color=colors, width=0.6)
    for bar, s in zip(bars, scores):
        delta = s - baseline_score
        ax.text(bar.get_x() + bar.get_width() / 2, s + 0.015,
                f"{s:.0%}\n({delta * 100:+.0f} pts)",
                ha="center", va="bottom", fontsize=9)

    baseline = ax.axhline(baseline_score, color=PURPLE, linestyle="--", linewidth=1.5)
    legend = [(baseline, f"Initial prompt ({baseline_score:.0%})")]
    for color, label in [(GREEN, "Better than initial"), (AMBER, "Same as initial"),
                         (RED, "Worse than initial")]:
        if color in colors:
            legend.append((Patch(color=color), label))
    ax.legend(*zip(*legend), loc="upper left", ncol=len(legend), frameon=False, fontsize=9)

    ax.set_ylim(0, 1.18)
    ax.yaxis.set_major_formatter(PercentFormatter(1.0))
    ax.set_ylabel("Accuracy on the optimization examples")
    ax.set_title("Round 1: how does each candidate compare to the initial prompt?")
    plt.tight_layout(); plt.show()


plot_candidates_vs_baseline(candidate_results, initial_score)

### Reading the result

The dashed line is the prompt we started with; each bar is one candidate the optimizer wrote. In our run, all four candidates beat the initial prompt, by 14 to 19 points, even though they all came from the *same* meta-prompt. They still differ from each other, which is exactly why we generate several and keep only the best.

Your run may also produce candidates that score *below* the initial prompt (red bars). That is normal and not wasted: every candidate and its score goes into the history, which shows the optimizer what *not* to do in the next round.

### Inspect the candidate prompts

In [ ]:
# @title 📊 Visualization: inspect the candidate prompts (double-click to view the code) { display-mode: "form" }
import difflib
import html
import re
import uuid
from IPython.display import HTML, display


def highlight_additions(reference, text):
    """Returns `text` as HTML, with words that are not in `reference` highlighted.

    Args:
        reference: The prompt to compare against.
        text: The prompt to display.

    Returns:
        An HTML string.
    """
    tokens = lambda s: re.findall(r"\s+|\S+", s.strip())
    a, b = tokens(reference), tokens(text)
    parts = []
    for op, _, _, j1, j2 in difflib.SequenceMatcher(None, a, b, autojunk=False).get_opcodes():
        chunk = "".join(b[j1:j2])
        if op != "equal" and chunk.strip():
            parts.append(f"<ins>{html.escape(chunk)}</ins>")
        else:
            parts.append(html.escape(chunk))
    return "".join(parts)


def render_candidates(candidate_results, reference_prompt, reference_score):
    """Builds one collapsible card per candidate prompt, best candidate opened.

    Args:
        candidate_results: List of dicts with "candidate_number", "prompt" and "score".
        reference_prompt: The prompt the candidates are compared against.
        reference_score: Accuracy of the reference prompt.

    Returns:
        The HTML string for the candidate cards.
    """
    uid = "ci" + uuid.uuid4().hex[:8]
    top = max(r["score"] for r in candidate_results)
    cards = []
    for r in candidate_results:
        delta = r["score"] - reference_score
        tone = "good" if delta > 0 else "bad" if delta < 0 else "same"
        is_top = r["score"] == top
        first_line = next((l for l in r["prompt"].splitlines() if l.strip()), "")
        cards.append(f"""
  <details class="cand{' top' if is_top else ''}"{' open' if is_top else ''}>
    <summary>
      <span class="name">Candidate {r['candidate_number']}{' 🏆' if is_top else ''}</span>
      <span class="pill {tone}">{r['score']:.0%} · {delta * 100:+.0f} pts</span>
      <span class="preview">{html.escape(first_line[:70])}</span>
    </summary>
    <pre>{highlight_additions(reference_prompt, r['prompt'])}</pre>
  </details>""")

    return f"""
<div id="{uid}">
<style>
  #{uid} {{ font-family: system-ui, 'Segoe UI', Roboto, sans-serif; color: #2b2d42;
           border-radius: 18px; border: 1px solid #ecebff; padding: 16px 18px;
           background: linear-gradient(135deg, #f6f8ff, #fbf5ff); }}
  #{uid} .sub {{ margin: 0 0 12px 0; font-size: 13px; color: #5c5f77; line-height: 1.45; }}
  #{uid} .cand {{ background: #fff; border: 1px solid #e6e8ee; border-radius: 14px;
                 margin-bottom: 8px; padding: 0 14px; }}
  #{uid} .cand.top {{ border: 2px solid {GREEN}; }}
  #{uid} summary {{ cursor: pointer; padding: 10px 0; display: flex; flex-wrap: wrap;
                   align-items: center; gap: 10px; }}
  #{uid} .name {{ font-weight: 700; font-size: 14px; }}
  #{uid} .pill {{ font-size: 12px; font-weight: 700; border-radius: 999px; padding: 2px 10px; }}
  #{uid} .pill.good {{ background: {GREEN}26; color: #1d6b3d; }}
  #{uid} .pill.bad {{ background: {RED}33; color: #8f2f24; }}
  #{uid} .pill.same {{ background: {AMBER}33; color: #8a5a00; }}
  #{uid} .preview {{ font-size: 12px; color: #8a8da8; flex: 1 1 200px; overflow: hidden;
                    white-space: nowrap; text-overflow: ellipsis; }}
  #{uid} pre {{ margin: 0 0 12px 0; white-space: pre-wrap; word-break: break-word;
               font-size: 12.5px; line-height: 1.5; background: none; color: #2b2d42;
               font-family: ui-monospace, Menlo, Consolas, monospace; }}
  #{uid} ins {{ background: {GREEN}33; color: #1d6b3d; text-decoration: none;
               border-radius: 3px; font-weight: 600; }}
</style>
  <p class="sub">Click a candidate to open it. Points are compared with the initial prompt
  ({reference_score:.0%}); <ins>highlighted</ins> words are new compared with the initial prompt.
  The highest-scoring candidate is opened for you.</p>
  {"".join(cards)}
</div>
"""


display(HTML(render_candidates(candidate_results, initial_prompt, initial_score)))

### 🎯 Keep the best candidate

We keep the highest-scoring candidate.

A lower-scoring candidate is not a failure of the algorithm. It is simply part of the search process.

In [ ]:
for result in candidate_results:
    if result["score"] > best_score:
        #🎯TODO: complete these lines:
        #Solution starts here
        best_prompt = result["prompt"]
        best_score = result["score"]
        #Solution ends here

print(
    "Best score after round 1:",
    f"{best_score:.0%}"
)

## Part 6 — The full OPRO loop: does the search keep improving?

### 🎯 The full OPRO-style optimization loop

Each round does six things:

1. Read the current optimization history.
2. Generate several candidates.
3. Evaluate every candidate.
4. Add every candidate and its score to the history.
5. Keep the best-so-far prompt.
6. Repeat.

This is the central exercise.

In [ ]:
def opro(
    initial_prompt,
    dataset,
    rounds=2,
    candidates_per_round=4
):

    initial_score = evaluate(initial_prompt,dataset)
    history = [{
            "round": 0,
            "candidate_number": 0,
            "prompt": initial_prompt,
            "score": initial_score}]

    best_prompt = initial_prompt
    best_score = initial_score

    for round_number in range(1,rounds + 1):

        print()
        print("=" * 70)
        print(f"ROUND {round_number}")
        print("=" * 70)
        print(
            "Best-so-far:",
            f"{best_score:.0%}")

        candidates = generate_candidates(
            history,
            best_prompt,
            best_score,
            number_of_candidates=candidates_per_round
        )

        if not candidates:
            print(
                "No valid candidates were generated.")
            continue

        for candidate_number, candidate in enumerate(candidates,start=1):
            #🎯TODO: Get the score for each candidate
            #Hint: look at how evaluate function works
            score = evaluate(candidate,dataset)
            history.append({
                "round": round_number,
                "candidate_number": candidate_number,
                "prompt": candidate,
                "score": score})

            print(
                f"Candidate {candidate_number}: "
                f"{score:.0%}"
            )
            #🎯TODO: Complete the following if statement
            #Hint: when should best_score=score
            if score > best_score:
                best_prompt = candidate
                best_score = score

                print(
                    "  ✓ NEW BEST:",
                    f"{best_score:.0%}")

        print(
            "Best-so-far after round:",
            f"{best_score:.0%}")

    return (best_prompt,best_score,history)

### 🎯 Run OPRO

For a first test, use:

- 2 rounds
- 3 candidates per round
Feel free to play around with the number of rounds

In [ ]:
#🎯TODO: call the OPRO function defined above
#Hint: set rounds equal to 2 and candidates_per_round equal to 3
best_prompt, best_score, history = opro(
    initial_prompt,
    optimization_examples,
    rounds=5,
    candidates_per_round=3
)

print()
print(
    "FINAL BEST SCORE:",
    f"{best_score:.0%}"
)

### Inspect the optimization trajectory

The plot below shows the whole search at once: **every** candidate the optimizer proposed, and the **best prompt found so far** — the one you would actually put into production.

In [ ]:
# @title 📊 Visualization: every candidate vs. the best-so-far prompt (double-click to view the code) { display-mode: "form" }
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

GREY = "#b8bccf"


def plot_search(history):
    """Plots every prompt in evaluation order, grouped by round, with the best-so-far line.

    Args:
        history: List of dicts with "round", "candidate_number" and "score",
            in the order they were evaluated (as returned by opro()).
    """
    fig, ax = plt.subplots(figsize=(9, 4.8))
    x, best = 0, -1
    xs, best_line, tick_labels = [], [], []

    for r in sorted(set(item["round"] for item in history)):
        items = [item for item in history if item["round"] == r]
        start = x
        for item in items:
            is_new_best = item["score"] > best
            best = max(best, item["score"])
            if r == 0:
                color, label, tick = PURPLE, "Initial prompt", "Initial"
            elif is_new_best:
                color, label, tick = GREEN, "Candidate — new best", f"C{item['candidate_number']}"
            else:
                color, label, tick = GREY, "Candidate — not better", f"C{item['candidate_number']}"
            ax.scatter(x, item["score"], s=80, color=color, edgecolor="white",
                       linewidth=1, zorder=3, label=label)
            ax.annotate(f"{item['score']:.0%}", (x, item["score"]), xytext=(0, 8),
                        textcoords="offset points", ha="center", fontsize=8.5, color="#4a4d63")
            xs.append(x); best_line.append(best); tick_labels.append(tick)
            x += 1
        # shade every round and name it under the axis
        if r > 0:
            ax.axvspan(start - 0.5, x - 0.5, color="#f3f4fb" if r % 2 else "#fbf5ff", zorder=0)
        ax.text((start + x - 1) / 2, -0.16, "Start" if r == 0 else f"Round {r}",
                transform=ax.get_xaxis_transform(), ha="center", va="top",
                fontsize=10, fontweight="bold", color="#2b2d42")

    ax.step(xs + [xs[-1] + 0.5], best_line + [best_line[-1]], where="post",
            color=GREEN, linewidth=2.2, zorder=2, label="Best-so-far prompt")
    ax.axhline(history[0]["score"], color=PURPLE, linestyle="--", linewidth=1, alpha=0.6)

    handles, labels = ax.get_legend_handles_labels()
    unique = dict(zip(labels, handles))
    ax.legend(unique.values(), unique.keys(), loc="lower right", frameon=False, fontsize=9)

    ax.set_xticks(xs, tick_labels)
    ax.set_xlim(-0.5, xs[-1] + 0.5)
    ax.set_ylim(max(0, min(item["score"] for item in history) - 0.15), 1.05)
    ax.yaxis.set_major_formatter(PercentFormatter(1.0))
    ax.set_ylabel("Accuracy on the optimization examples")
    ax.set_title("OPRO search: every prompt tried, in order, and the best prompt so far")
    ax.grid(axis="y", color="#e6e8ee")
    plt.tight_layout(); plt.show()


plot_search(history)

### Reading the result

Read the chart from left to right: each dot is one prompt, in the order it was evaluated — first the initial prompt, then the candidates of round 1 (C1, C2, C3), then those of round 2. The shaded bands group the rounds. A grey dot is a candidate that did **not** beat the best prompt at the time; a green dot is one that did. The green line is the best-so-far prompt, and it can only go up or stay flat — we never replace the deployed prompt with a worse one.

In our run, round 1 lifted accuracy from 59% to 89%, and none of the round-2 candidates beat that, so the line stays flat. Your exact numbers may differ. The point is **not** that every candidate improves. The point is:

> **The optimizer explores multiple candidate prompts, the evaluator scores them, and the best-so-far solution is carried forward.**

For the business, a flat round costs compute time, never quality. Each round costs *(candidates × examples)* model calls, so a line that has stopped climbing is your signal that further rounds are no longer paying off.

### Inspect the final optimized prompt

In [ ]:
print(best_prompt)

In [ ]:
# @title 📊 Visualization: what did the optimizer change? (double-click to view the code) { display-mode: "form" }
import difflib
import html
import re
import uuid
from IPython.display import HTML, display


def render_prompt_diff(old, new, old_title, new_title):
    """Renders two prompts side by side with word-level changes highlighted.

    Args:
        old: The original prompt.
        new: The changed prompt.
        old_title: Heading for the original prompt's panel.
        new_title: Heading for the changed prompt's panel.

    Returns:
        The HTML string for the side-by-side comparison.
    """
    tokens = lambda text: re.findall(r"\s+|\S+", text.strip())
    a, b = tokens(old), tokens(new)
    left, right, removed, added = [], [], 0, 0
    for op, i1, i2, j1, j2 in difflib.SequenceMatcher(None, a, b, autojunk=False).get_opcodes():
        old_part, new_part = "".join(a[i1:i2]), "".join(b[j1:j2])
        if op == "equal":
            left.append(html.escape(old_part)); right.append(html.escape(new_part))
            continue
        if old_part.strip():
            left.append(f"<del>{html.escape(old_part)}</del>"); removed += len(old_part.split())
        else:
            left.append(html.escape(old_part))
        if new_part.strip():
            right.append(f"<ins>{html.escape(new_part)}</ins>"); added += len(new_part.split())
        else:
            right.append(html.escape(new_part))

    uid = "pd" + uuid.uuid4().hex[:8]
    return f"""
<div id="{uid}">
<style>
  #{uid} {{ font-family: system-ui, 'Segoe UI', Roboto, sans-serif; color: #2b2d42;
           border-radius: 18px; border: 1px solid #ecebff; padding: 16px 18px;
           background: linear-gradient(135deg, #f6f8ff, #fbf5ff); }}
  #{uid} .row {{ display: flex; flex-wrap: wrap; gap: 12px; }}
  #{uid} .panel {{ flex: 1 1 340px; background: #fff; border: 1px solid #e6e8ee;
                  border-radius: 14px; padding: 12px 14px; min-width: 0; }}
  #{uid} h4 {{ margin: 0 0 8px 0; font-size: 14.5px; }}
  #{uid} pre {{ margin: 0; white-space: pre-wrap; word-break: break-word; font-size: 12.5px;
               line-height: 1.5; font-family: ui-monospace, Menlo, Consolas, monospace;
               background: none; color: #2b2d42; }}
  #{uid} del {{ background: {RED}33; color: #8f2f24; text-decoration: line-through;
               border-radius: 3px; }}
  #{uid} ins {{ background: {GREEN}33; color: #1d6b3d; text-decoration: none;
               border-radius: 3px; font-weight: 600; }}
  #{uid} .legend {{ margin-top: 10px; font-size: 12.5px; color: #5c5f77; }}
</style>
  <div class="row">
    <div class="panel"><h4>{html.escape(old_title)}</h4><pre>{"".join(left)}</pre></div>
    <div class="panel"><h4>{html.escape(new_title)}</h4><pre>{"".join(right)}</pre></div>
  </div>
  <div class="legend"><del>struck-through</del> = removed by the optimizer ({removed} words) ·
  <ins>highlighted</ins> = added by the optimizer ({added} words) · everything else is unchanged.</div>
</div>
"""


display(HTML(render_prompt_diff(
    initial_prompt, best_prompt,
    f"Initial prompt — {initial_score:.0%}",
    f"Optimized prompt — {best_score:.0%}",
)))

### Reading the result

In our run, the optimizer kept the prompt's structure — the task sentence, the `{message}` slot and the output instruction are untouched — and rewrote **only the department descriptions**. The vague labels ("Technical issues", "Product and feature questions") were replaced by precise definitions that closely follow the task specification it was shown in the meta-prompt.

That is exactly where Part 3 located the damage: D7 and D8 were never routed correctly. The win did not come from a clever trick; it came from telling the model what each team actually handles — the same thing a good onboarding document for a human support agent would do.

Finally, remember that this prompt was written by a machine and chosen by a score on 64 examples. In practice you would still **review an optimized prompt before deploying it**: does it say what the business intends, does it contain anything odd, and does it hold up on unseen tickets (Part 7)? Treat it exactly like any other change to a business process.

## Part 7 — Generalization: does the prompt work on tickets it has never seen?

The optimization examples were used during the search.

Now we test the final prompt on examples that the optimizer has never seen.

This helps us see whether the optimized prompt generalizes.

In [ ]:
test_examples = [

    # D1 — Duplicate or unexpected charge
    ("My bank statement shows the same purchase two times.", "D1"),
    ("I don't remember authorizing this additional payment.", "D1"),
    ("Why was I billed twice when I only placed one order?", "D1"),
    ("There is an unexpected second transaction for my purchase.", "D1"),

    # D2 — Refund
    ("The money from my return hasn't come back yet.", "D2"),
    ("I'd like to have the payment for this order returned.", "D2"),
    ("You said the reimbursement was sent, but I still can't see it.", "D2"),
    ("Could you tell me when I should expect my refund?", "D2"),

    # D3 — Delivery
    ("The expected delivery date has passed and I'm still waiting.", "D3"),
    ("My parcel appears to be stuck and hasn't reached me.", "D3"),
    ("I ordered this a while ago but nothing has arrived.", "D3"),
    ("Can someone check what is happening with my delayed shipment?", "D3"),

    # D4 — Wrong/damaged item
    ("The parcel arrived, but the item inside isn't what I selected.", "D4"),
    ("My new device arrived with a cracked screen.", "D4"),
    ("I received the wrong size instead of the one I ordered.", "D4"),
    ("The package was delivered, but the product is damaged.", "D4"),

    # D5 — Login/password
    ("My credentials aren't being accepted when I try to sign in.", "D5"),
    ("I can't remember the password I used for my account.", "D5"),
    ("I have been locked out after trying to log in.", "D5"),
    ("The sign-in process won't let me access my account.", "D5"),

    # D6 — Account information
    ("I got a new phone and need to change the number on my profile.", "D6"),
    ("Where can I correct the address listed in my account?", "D6"),
    ("I need to change some personal details in my profile.", "D6"),
    ("How can I replace the email address on my account?", "D6"),

    # D7 — Technical problem
    ("The checkout page stops responding when I click the payment button.", "D7"),
    ("The mobile application closes unexpectedly whenever I open it.", "D7"),
    ("I can't upload my document because the site keeps failing.", "D7"),
    ("The service isn't working correctly and displays an error.", "D7"),

    # D8 — Feature help
    ("How can I enable notifications for new messages?", "D8"),
    ("What steps do I follow to use the reporting function?", "D8"),
    ("Can you explain how the filtering options work?", "D8"),
    ("Where do I find the option for changing the application's appearance?", "D8"),

    # ---------------------------------------------------------
    # HARD EXAMPLES

    # D1 vs D2 — duplicate charge is the primary issue
    ("I was charged for an order and now I want the money back because I was billed twice.", "D1"),

    # D3 vs D4 — delivery happened, but the item is wrong
    ("My order finally arrived, but the item inside isn't the one I ordered.", "D4"),

    # D5 vs D7 — login problem is caused by a technical failure
    ("I can't log in because the sign-in page keeps crashing.", "D7"),

    # D6 vs D7 — technical problem prevents changing account information
    ("The website crashes when I try to change my email address.", "D7"),

    # D2 vs D3 — the primary issue is that the order never arrived
    ("My order never arrived and I want my money returned.", "D3"),

    # D4 vs D2 — damaged product is the primary issue
    ("The product arrived broken and I want my money back.", "D4"),

    # D8 vs D7 — primarily asking how to use a feature
    ("I don't know how to use the export button and it also seems to be giving me an error.", "D8"),

    # D6 vs D5 — inability to change information is blocked by login
    ("I can't change the email on my account because I can't sign in.", "D5"),
]

In [ ]:
initial_test_score, initial_test_details = evaluate(initial_prompt,test_examples,return_details=True)

optimized_test_score, optimized_test_details = evaluate(best_prompt,test_examples,return_details=True)

print(
    "Initial prompt — test accuracy:",
    f"{initial_test_score:.0%}")

print(
    "Optimized prompt — test accuracy:",
    f"{optimized_test_score:.0%}")

In [ ]:
# @title 📊 Visualization: initial vs. optimized prompt on unseen tickets (double-click to view the code) { display-mode: "form" }
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

N_HARD = 8  # the last 8 test examples are the deliberately ambiguous "hard" cases


def plot_test_comparison(initial_details, optimized_details, n_hard):
    """Plots initial vs. optimized accuracy on all, normal and hard test tickets.

    Args:
        initial_details: Per-message results of the initial prompt on the test set.
        optimized_details: Per-message results of the optimized prompt on the test set.
        n_hard: Number of hard examples at the end of the test set.
    """
    accuracy = lambda details: sum(d["correct"] for d in details) / len(details)
    n_normal = len(initial_details) - n_hard
    groups = [
        (f"All test tickets ({len(initial_details)})", slice(None)),
        (f"Normal tickets ({n_normal})", slice(None, n_normal)),
        (f"Hard, ambiguous tickets ({n_hard})", slice(n_normal, None)),
    ]
    x = np.arange(len(groups))
    width = 0.36

    fig, ax = plt.subplots(figsize=(8.5, 4.4))
    for offset, details, color, label in [
        (-width / 2, initial_details, "#b8bccf", "Initial prompt"),
        (width / 2, optimized_details, PURPLE, "Optimized prompt"),
    ]:
        values = [accuracy(details[s]) for _, s in groups]
        bars = ax.bar(x + offset, values, width, color=color, label=label)
        for bar, v in zip(bars, values):
            ax.text(bar.get_x() + bar.get_width() / 2, v + 0.015, f"{v:.0%}",
                    ha="center", va="bottom", fontsize=9)

    ax.set_xticks(x, [name for name, _ in groups])
    ax.set_ylim(0, 1.12)
    ax.yaxis.set_major_formatter(PercentFormatter(1.0))
    ax.set_ylabel("Accuracy on unseen test tickets")
    ax.set_title("Does the optimized prompt hold up on tickets it has never seen?")
    ax.legend(loc="upper right", frameon=False, fontsize=9, ncol=2)
    plt.tight_layout(); plt.show()


plot_test_comparison(initial_test_details, optimized_test_details, N_HARD)

# Same report as in Part 3, now for the optimized prompt on the unseen test tickets
plot_routing_report(
    optimized_test_details,
    f"Optimized prompt on unseen tickets — {optimized_test_score:.0%} routed correctly",
)

## Summary

### What each OPRO component means

| Component | In this exercise |
|---|---|
| Solution | The prompt |
| Optimizer | The optimizer LLM |
| Objective | Classification accuracy |
| Evaluation | Run the prompt on examples |
| Feedback | Previous prompts + their scores |
| Search | Generate multiple candidates |
| Best solution | Highest-scoring prompt so far |

The optimizer is **not changing model weights**.

It is generating new candidate solutions.

### Why did we deliberately start with a weak prompt?

An optimization exercise needs **room for improvement**: if the initial prompt already scored 100%, there would be nothing to optimize.

That is why the initial prompt names each department with only a vague label ("Payment-related issues", "Technical issues") instead of the precise definitions from the task specification. The optimizer *is* shown those precise definitions in its meta-prompt, so it has information the initial prompt lacks, and Part 6 showed that it found and used it.

In a real project, the starting point is usually your team's current prompt. OPRO is then a way to improve it systematically instead of by trial and error.

### Reflection questions

#### Who creates the new prompt?

The **optimizer LLM**.

#### Who scores the candidate?

The **evaluation function**.

#### What is being optimized?

The **prompt**.

#### What is the objective?

In this exercise:

> **maximize classification accuracy**

#### Does every candidate improve?

No. Some candidates are worse.

#### Why generate multiple candidates?

Because optimization is a search process.

#### Why keep a history?

The optimizer can use previous solutions and their scores to guide later proposals.

#### Why use a separate test set?

To see whether the final prompt works on new examples.

### Final takeaway

> **OPRO uses an LLM to generate candidate solutions, and the objective scores of previous candidates to guide the further search.**

The diagram below puts the whole notebook into one picture, and maps each OPRO term onto what it was in our routing problem.

In [ ]:
# @title 🧩 Visualization: the OPRO loop in one picture (double-click to view the code) { display-mode: "form" }
import uuid
from IPython.display import HTML, display

# (icon, label, what happens)
LOOP_STEPS = [
    ("🤖", "Optimizer LLM", "reads the meta-prompt: rules, history, best so far"),
    ("📝", "Candidates", "several new prompts per round"),
    ("🧪", "Evaluate", "accuracy on labelled examples"),
    ("🏆", "Keep the best", "and add every candidate + score to the history"),
]
# (OPRO term, what it was in this notebook)
MAPPING = [
    ("Solution", "the prompt"),
    ("Optimizer", "an LLM writing new prompts"),
    ("Objective", "routing accuracy"),
    ("Feedback", "previous prompts + their scores"),
]


def render_takeaway(loop_steps, mapping):
    """Builds the final-takeaway diagram as an HTML string.

    Args:
        loop_steps: List of (icon, label, blurb) for the steps inside the loop.
        mapping: List of (OPRO term, meaning in this notebook) pairs.

    Returns:
        The HTML string for the diagram.
    """
    uid = "tk" + uuid.uuid4().hex[:8]
    arrow = '<div class="arrow">→</div>'
    node = lambda icon, label, blurb, cls="": f"""
        <div class="node {cls}"><div class="dot">{icon}</div>
          <div class="label">{label}</div><div class="blurb">{blurb}</div></div>"""
    loop = arrow.join(node(*s) for s in loop_steps)
    cards = "".join(f'<div class="map"><div class="term">{t}</div><div class="means">= {m}</div></div>'
                    for t, m in mapping)

    return f"""
<div id="{uid}">
<style>
  #{uid} {{ font-family: system-ui, 'Segoe UI', Roboto, sans-serif; color: #2b2d42;
           border-radius: 18px; border: 1px solid #ecebff; padding: 18px 20px;
           background: linear-gradient(135deg, #f6f8ff, #fbf5ff); max-width: 900px; }}
  #{uid} h3 {{ margin: 0 0 14px 0; font-size: 18px; }}
  #{uid} .flow {{ display: flex; flex-direction: column; align-items: center; gap: 6px; }}
  #{uid} .loop {{ align-self: stretch; display: flex; flex-wrap: wrap; align-items: stretch; gap: 8px;
                 position: relative; border: 2px dashed {PURPLE}; border-radius: 16px;
                 padding: 24px 10px 10px 10px; background: rgba(102,126,234,.07); }}
  #{uid} .tag {{ position: absolute; top: -12px; left: 12px; padding: 3px 12px; color: #fff;
                border-radius: 999px; font-size: 12px; font-weight: 700;
                background: linear-gradient(135deg, {PURPLE}, {PURPLE2}); }}
  #{uid} .repeat {{ position: absolute; top: 4px; right: 12px; font-size: 12px; font-weight: 700;
                   color: {PURPLE2}; }}
  #{uid} .node {{ flex: 1 1 110px; max-width: 170px; background: #fff; border: 1px solid #e6e8ee;
                 border-radius: 14px; padding: 10px 8px; text-align: center; box-sizing: border-box; }}
  #{uid} .node.ends {{ flex: none; width: 100%; max-width: 300px; }}
  #{uid} .node.deploy {{ border: 2px solid {GREEN}; }}
  #{uid} .dot {{ width: 40px; height: 40px; margin: 0 auto 4px auto; border-radius: 50%;
                font-size: 19px; display: flex; align-items: center; justify-content: center;
                background: linear-gradient(135deg, {PURPLE}, {PURPLE2}); }}
  #{uid} .label {{ font-weight: 700; font-size: 13.5px; }}
  #{uid} .blurb {{ font-size: 11.5px; color: #5c5f77; line-height: 1.35; margin-top: 2px; }}
  #{uid} .arrow {{ align-self: center; color: {PURPLE}; font-size: 18px; font-weight: 700; }}
  #{uid} .down {{ color: {PURPLE}; font-size: 16px; font-weight: 700; }}
  #{uid} .maps {{ display: flex; flex-wrap: wrap; gap: 8px; margin-top: 16px; }}
  #{uid} .map {{ flex: 1 1 160px; background: #fff; border: 1px solid #e6e8ee; border-radius: 14px;
                padding: 10px 12px; }}
  #{uid} .term {{ font-size: 11px; font-weight: 700; letter-spacing: .06em; text-transform: uppercase;
                 color: {PURPLE2}; }}
  #{uid} .means {{ font-size: 14px; font-weight: 600; margin-top: 2px; }}
</style>
  <h3>🧩 OPRO in one picture</h3>
  <div class="flow">
    {node("✍️", "Initial prompt", "a simple, hand-written start", "ends")}
    <div class="down">↓</div>
    <div class="loop"><span class="tag">OPRO loop</span><span class="repeat">↻ repeat</span>{loop}</div>
    <div class="down">↓ best prompt found</div>
    {node("🧭", "Test, review, deploy", "check on unseen tickets before going live", "ends deploy")}
  </div>
  <div class="maps">{cards}</div>
</div>
"""


display(HTML(render_takeaway(LOOP_STEPS, MAPPING)))